# OmniDrop — Walkthrough

Reproduction of **arXiv 2605.14458**, *"OmniDrop: Layer-wise Token Pruning for Omni-modal LLMs via Query-Guidance"*.

This notebook connects each paper section to the code with runnable sanity checks. Run top-to-bottom.

> Pure NumPy so it runs without the Qwen2.5-Omni stack. The same index/mask logic plugs into the real decoder forward pass.


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import numpy as np
import omnidrop as od
print('omnidrop', od.__version__)

## §3.1 — Progressive Layer-wise Pruning (Eqs. 3–4)

`p_l = p_init + (p_final − p_init)·σ(β·(l/L − t_mid))`, applied up to layer `L−2`.


In [ ]:
from omnidrop import PLPSchedule
sch = PLPSchedule(p_init=0.0, p_final=0.2, L=28, t_mid=0.5, beta=20.0)  # 7B / 30%
ratios = [round(sch.pruning_ratio(l), 4) for l in range(28)]
print('p_l by layer:', ratios)
print('layer 27 (L-1) capped to:', sch.pruning_ratio(27))
assert sch.pruning_ratio(27) == 0.0  # penultimate-layer cap

### Appendix-E headline check (a falsifiable number)

The paper derives `p_final ≈ 0.146` for a 30% mean. Our closed-form reproduces it; the **exact** solver shows the paper's Taylor approximation undershoots (exact ≈ 0.206 → see REPRODUCTION_NOTES).


In [ ]:
from omnidrop import calibrate_pfinal_paper_approx, calibrate_pfinal
approx = calibrate_pfinal_paper_approx(0.30, L=28, r0=0.45)
exact  = calibrate_pfinal(0.30, L=28, p_init=0.0, t_mid=0.5, beta=20.0, r0=0.45)
print(f'paper closed-form  p_final = {approx:.4f}  (target ~0.146)')
print(f'exact numeric      p_final = {exact:.4f}')
print(f'mean retained @0.146 = {PLPSchedule(0,0.146,28,0.5,20).mean_retained(0.45):.4f}  (NOT 0.30!)')
print(f'mean retained @0.2   = {PLPSchedule(0,0.2,28,0.5,20).mean_retained(0.45):.4f}')
assert abs(approx - 0.146) < 0.01

## §3.2 — Query-guided importance (Eq. 5)

`S_j = mean over text-query tokens of attention to AV token j`. Higher = more query-relevant.


In [ ]:
from omnidrop import query_guided_importance
seq = 6; attn = np.zeros((seq, seq))
text_idx = np.array([0, 1]); av_idx = np.array([2, 3, 4, 5])
attn[text_idx[:, None], av_idx[None, :]] = np.array([[0.1,0.5,0.9,0.0],
                                                     [0.3,0.5,0.1,0.2]])
S = query_guided_importance(attn, text_idx, av_idx)
print('importance S:', S)  # mean over the two text rows
assert np.allclose(S, [0.2, 0.5, 0.5, 0.1])

## §3.3 — Temporal Diversity Score (Algorithm 1)

Among the bottom-`2k` candidates, boost tokens temporally far from the key chunk so global context survives.


In [ ]:
from omnidrop import tds_select_to_prune
S = np.array([0.9, 0.10, 0.11, 0.12, 0.13, 0.05])
chunks = np.array([0, 1, 2, 3, 4, 5])  # key chunk = 0 (argmax S at idx 0)
no_div = tds_select_to_prune(S, chunks, k=1, lambda_div=0.0)
with_div = tds_select_to_prune(S, chunks, k=1, lambda_div=1.0)
print('prune w/o diversity:', no_div, '(raw lowest = distant idx5)')
print('prune w/  diversity:', with_div, '(distant idx5 spared; near idx1 dropped)')
assert no_div.tolist() == [5] and with_div.tolist() == [1]

## §3.4 — Intra-modality pre-LLM pruning

Audio: keep top-attention (70%). Video: Dycoke-TTM over 4-frame groups (40%). Combined ≈ 45%.


In [ ]:
from omnidrop import prune_audio_by_attention, prune_video_ttm
audio_keep = prune_audio_by_attention(np.arange(50.0), prune_ratio=0.3)
rng = np.random.default_rng(0)
video = rng.standard_normal((16, 288, 8))
video_keep = prune_video_ttm(video, group_size=4, prune_rate=0.8)
a = len(audio_keep)/50; v = len(video_keep)/(16*288)
print(f'audio retained {a:.2%}, video retained {v:.2%}')
print(f'combined (50A+288V): {(0.7*50+0.4*288)/338:.2%}')
assert abs(a-0.70)<1e-9 and abs(v-0.40)<0.01

## End-to-end — cumulative layer-wise pruning

Wire it together. `k_l` is computed on the **current** alive count each layer; dropped tokens never return.


In [ ]:
from omnidrop import OmniDropConfig, TokenLayout, OmniDropPruner
n_chunks, av_per_chunk, n_sys, n_text = 40, 10, 4, 8
pos = n_sys; av_idx=[]; av_chunk=[]
for c in range(n_chunks):
    for _ in range(av_per_chunk):
        av_idx.append(pos); av_chunk.append(c); pos += 1
text_query_idx = np.arange(pos, pos+n_text); seq_len = pos+n_text
layout = TokenLayout(text_query_idx, np.array(av_idx), np.array(av_chunk), n_chunks)
cfg = OmniDropConfig(p_init=0.0, p_final=0.2, L=28, tds_start_layer=14)
pruner = OmniDropPruner(cfg, layout)
rng = np.random.default_rng(1)
full = rng.random((seq_len, seq_len)); full /= full.sum(1, keepdims=True)
survivors = pruner.run(lambda layer: full)
counts = [len(s) for s in survivors]
print('alive AV tokens by layer:', counts)
print(f'mean retained (r0=0.45): {pruner.mean_retained(r0=0.45):.4f}  -> ~30%')
assert all(counts[i] >= counts[i+1] for i in range(len(counts)-1))  # monotone

## Summary

- **PLP** (§3.1) — sigmoid depth schedule, verified against Appendix E (and its approximation flagged).
- **Query guidance** (§3.2) — text→AV attention importance.
- **TDS** (§3.3) — temporal-diversity rescue of distant tokens.
- **Intra-modality** (§3.4) — 70% audio / 40% video / ~45% total.
- **Orchestrator** — cumulative, monotone, current-count budgeting.

See `REPRODUCTION_NOTES.md` for assumptions and the Appendix-E discrepancy.
